# 02 — Build features
No timestamps, so customer history = the TRAINING split. Training rows use leave-one-out, so a row never describes itself.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config

In [ ]:
from src.schema import load_clean_transactions
from src.train import split, featurize
from src.features import NUMERIC, CATEGORICAL

transactions, report = load_clean_transactions(config.RAW_PATH)
train_rows, validation_rows, test_rows = split(transactions)
print({name: (len(rows), round(rows.is_fraud.mean(), 4)) for name, rows in [("train", train_rows), ("validation", validation_rows), ("test", test_rows)]})
profiles, train_features, validation_features, test_features = featurize(train_rows, validation_rows, test_rows)
train_features[NUMERIC + CATEGORICAL + ["is_fraud"]].head()

## Leakage check
A training row's `customer_txn_count` is its customer's training count minus one.

In [ ]:
counts = train_rows.groupby("customer_id").size()
assert (train_features["customer_txn_count"] == train_features["customer_id"].map(counts) - 1).all()
assert (validation_features["customer_txn_count"] == validation_features["customer_id"].map(counts).fillna(0)).all()
print("leave-one-out and train-only profiles verified")

In [ ]:
profiles.profile(train_rows.customer_id.iloc[0])